In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

DATA_DIR = Path("..","data","CC_Force_Cards")   # searched recursively for .txt
OUTPUT_DIR = Path("..","outputs","grids.bdf")

COLUMNS  = ["Node ID", "x", "y", "z", "Fx", "Fy", "Fz"]

# Read Data

In [ ]:
frames = []
for f in sorted(DATA_DIR.rglob("*.txt")):
    d = pd.read_csv(f, skiprows=3, header=None, names=COLUMNS)
    d["loadcase"] = f.stem.split("_")[-2]
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
display(df)

In [ ]:
nodes_point_masses=[...]
df["is_point_mass"] = df["Node ID"].isin(nodes_point_masses)

In [ ]:
def nas_real(v, width=16):
    """Float -> Nastran-legal field of `width` chars. Nastran accepts an
    implied E, so 1.2346E+03 can be written 1.2346+3."""
    v = float(v)
    if v == 0.0:
        return "0.".ljust(width)

    def parse(t):
        t = t.strip()
        body = t[1:] if t[0] in "+-" else t
        for k, ch in enumerate(body):
            if ch in "+-":
                return float((t[0] if t[0] in "+-" else "") + body[:k] + "E" + body[k:])
        return float(t)

    cands = []
    for dec in range(width - 1, -1, -1):
        t = f"{v:.{dec}f}"
        if len(t) <= width and "." in t:
            cands.append(t); break
    for sig in range(width, 0, -1):
        mant, exp = f"{v:.{sig}E}".split("E")
        e = int(exp)
        mant = mant.rstrip("0").rstrip(".") if "." in mant else mant
        if "." not in mant: mant += "."
        c = f"{mant}{'+' if e >= 0 else '-'}{abs(e)}"
        if len(c) <= width:
            cands.append(c); break
    if not cands:
        raise ValueError(f"cannot fit {v!r} in {width} chars")
    return min(cands, key=lambda t: abs(parse(t) - v)).rjust(width)

In [ ]:
nodes = (df.drop_duplicates("Node ID")[["Node ID", "x", "y", "z"]]
           .sort_values("Node ID").reset_index(drop=True))

with open(OUTPUT_DIR, "w") as f:
    f.write(f"$ {len(nodes)} grid points from nodal load export\n")
    for r in nodes.itertuples(index=False):
        nid, x, y, z = int(r[0]), float(r[1]), float(r[2]), float(r[3])
        f.write(f"{'GRID*':<8}{nid:>16}{'':>16}{nas_real(x)}{nas_real(y)}*\n")
        f.write(f"{'*':<8}{nas_real(z)}{'':>16}\n")

# Calculate Section Loads

In [ ]:
ALLOWED_BODY_KEYS = {"nodes", "P0", "P1", "perp1", "stations"}
GLOBAL_COLS = ["Nx", "Vy", "Vz", "Mx", "My", "Mz"]

In [ ]:
def _validate(cfg):
    if not isinstance(cfg, dict) or not cfg:
        raise ValueError("CONFIG must be a non-empty dict of {body_name: spec}")
    if "bodies" in cfg and len(cfg) == 1:
        raise ValueError(
            "CONFIG still has its top-level 'bodies' wrapper. Load with "
            'json.loads(path.read_text())["bodies"].')

    for name, b in cfg.items():
        unknown = set(b) - ALLOWED_BODY_KEYS
        if unknown:
            raise ValueError(f"body '{name}': unknown key(s) {sorted(unknown)}. "
                             f"Allowed: {sorted(ALLOWED_BODY_KEYS)}.")
        missing = ALLOWED_BODY_KEYS - set(b)
        if missing:
            raise ValueError(f"body '{name}': missing {sorted(missing)}")

        P0, P1 = np.asarray(b["P0"], float), np.asarray(b["P1"], float)
        p1h = np.asarray(b["perp1"], float)
        for label, v in (("P0", P0), ("P1", P1), ("perp1", p1h)):
            if v.shape != (3,):
                raise ValueError(f"body '{name}': {label} must be 3 numbers")
        if np.linalg.norm(P1 - P0) <= 0:
            raise ValueError(f"body '{name}': P0 and P1 coincide, no axis")
        if np.linalg.norm(p1h) <= 0:
            raise ValueError(f"body '{name}': perp1 is a zero vector")

        e_s = (P1 - P0) / np.linalg.norm(P1 - P0)
        ang = np.degrees(np.arccos(
            min(abs(float(p1h @ e_s) / np.linalg.norm(p1h)), 1.0)))
        if ang < 5.0:
            raise ValueError(
                f"body '{name}': perp1 {p1h} is only {ang:.2f} deg from the axis, "
                "so the perpendicular frame is degenerate. For a near-vertical "
                "fin pass a lateral or chordwise direction, not [0,0,1].")
        if ang < 30.0:
            proj = p1h - (p1h @ e_s) * e_s
            print(f"  !! body '{name}': perp1 {p1h} is only {ang:.1f} deg from the "
                  f"axis. Projected it becomes "
                  f"{np.round(proj / np.linalg.norm(proj), 4)} -- dominated by a "
                  "small residual and sensitive to the axis direction.")

        st = b["stations"]
        mode = st.get("mode")
        if mode not in ("fraction", "interval"):
            raise ValueError(f"body '{name}': stations.mode must be "
                             "'fraction' or 'interval'")
        if mode == "interval" and float(st.get("step", 0)) <= 0:
            raise ValueError(f"body '{name}': stations.step must be > 0")
        if mode == "fraction" and "values" not in st and int(st.get("n", 0)) < 2:
            raise ValueError(f"body '{name}': stations.n must be >= 2")


def _axis(P0, P1, perp1_hint):
    """Axis frame. perp1_hint is projected square to the axis, so it need not
    already be perpendicular. Third axis = e_s x e_n."""
    P0, P1 = np.asarray(P0, float), np.asarray(P1, float)
    v = P1 - P0
    L = float(np.linalg.norm(v))
    e_s = v / L

    h = np.asarray(perp1_hint, float)
    e_n = h - (h @ e_s) * e_s
    e_n /= np.linalg.norm(e_n)
    e_c = np.cross(e_s, e_n)
    e_c /= np.linalg.norm(e_c)
    return e_s, e_c, e_n, L


def _stations(spec, L):
    if spec["mode"] == "interval":
        return np.arange(0.0, L + 1e-9, float(spec["step"]))
    if "values" in spec:
        f = np.asarray(spec["values"], float)
        if f.min() < -1e-9 or f.max() > 1 + 1e-9:
            raise ValueError("stations.values must lie in [0, 1]")
        return f * L
    return np.linspace(0.0, 1.0, int(spec["n"])) * L


def _select(df, spec, name=""):
    if spec.get("all"):
        return np.ones(len(df), bool)
    if "ids" in spec:
        return df["Node ID"].isin(np.asarray(spec["ids"])).to_numpy()
    col = spec.get("column")
    if col is None:
        raise ValueError(f"body '{name}': nodes needs 'column', 'ids', or all=true")
    if col not in df.columns:
        raise KeyError(f"body '{name}': nodes column '{col}' not in dataframe; "
                       f"have {list(df.columns)}")
    if "values" in spec:
        return df[col].isin(spec["values"]).to_numpy()
    if df[col].dtype != bool:
        raise ValueError(f"body '{name}': column '{col}' is not boolean; "
                         "supply 'values' as well")
    return df[col].to_numpy()

In [ ]:
def section_loads(df, cfg, case_col="loadcase"):
    """Section loads about a fixed P0->P1 axis datum, per body and loadcase."""
    _validate(cfg)
    out = []

    for name, b in cfg.items():
        P0 = np.asarray(b["P0"], float)
        e_s, e_c, e_n, L = _axis(P0, b["P1"], b["perp1"])
        stations = _stations(b["stations"], L)

        sub = df[_select(df, b["nodes"], name)]
        if sub.empty:
            raise ValueError(f"body '{name}': node selection matched nothing")

        X = sub[["x", "y", "z"]].to_numpy(float)
        F = sub[["Fx", "Fy", "Fz"]].to_numpy(float)
        M = (sub[["Mx", "My", "Mz"]].to_numpy(float)
             if {"Mx", "My", "Mz"} <= set(sub.columns) else np.zeros_like(F))
        sv = (X - P0) @ e_s
        cases = (sub[case_col].to_numpy() if case_col in sub.columns
                 else np.array(["ALL"] * len(sub)))

        # Nudge stations off node planes: a strict inequality would drop an
        # entire plane of nodes. eps is a quarter of the tightest node gap, so
        # a nudged station can never step past a neighbouring plane.
        planes = np.unique(np.round(sv, 9))
        moved, eps = [], 0.0
        if len(planes) > 1:
            eps = max(float(np.diff(planes).min()) * 0.25, 1e-6)
            keep = []
            for s in stations:
                if np.isclose(planes, s, atol=1e-9).any():
                    c = s + eps
                    if c > L + 1e-9 or np.isclose(planes, c, atol=1e-9).any():
                        c = s - eps
                    moved.append((s, c))
                    keep.append(c)
                else:
                    keep.append(s)
            stations = np.array(keep)

        for case in pd.unique(cases):
            cm = cases == case
            for s in stations:
                m = cm & (sv < s - 1e-9)          # P0-side free body
                P = P0 + e_s * s                  # moment reference on the axis
                if m.any():
                    Ft = F[m].sum(axis=0)
                    # nodal moments add directly -- a couple has no lever arm
                    Mt = np.cross(X[m] - P, F[m]).sum(axis=0) + M[m].sum(axis=0)
                else:
                    Ft, Mt = np.zeros(3), np.zeros(3)
                out.append({
                    "body": name, "loadcase": case, "station": s,
                    "station_frac": s / L, "n_free_body": int(m.sum()),
                    "ref_x": P[0], "ref_y": P[1], "ref_z": P[2],
                    "Nx": Ft[0], "Vy": Ft[1], "Vz": Ft[2],
                    "Mx": Mt[0], "My": Mt[1], "Mz": Mt[2],
                    "N_axis": Ft @ e_s,          # along the axis
                    "V_perp1": Ft @ e_n,         # along e_n
                    "V_perp2": Ft @ e_c,         # along e_c
                    "T_axis": Mt @ e_s,          # torsion about the axis
                    "M_perp1": Mt @ e_n,         # about e_n, driven by V_perp2
                    "M_perp2": Mt @ e_c,         # about e_c, driven by V_perp1
                })

    return (pd.DataFrame(out)
              .set_index(["body", "loadcase", "station"]).sort_index())

In [ ]:
def check(res):
    rows = []
    for (body, case), g in res.groupby(level=["body", "loadcase"]):
        g = g.reset_index().sort_values("station")
        first = g.iloc[0]
        s = g["station"].to_numpy()
        if len(s) > 3:
            dM = np.gradient(g["M_perp2"].to_numpy(), s)
            V = g["V_perp1"].to_numpy()
            den = max(np.abs(V).max(), 1e-12)
            e_minus = np.abs(dM - V).max() / den
            e_plus = np.abs(dM + V).max() / den
        else:
            e_minus = e_plus = np.nan
        vsc = max(np.abs(g["V_perp1"]).max(), 1e-12)
        rows.append({
            "body": body, "loadcase": case,
            "first_station": first["station"],
            "first_resid": np.abs(first[GLOBAL_COLS].to_numpy(float)).max(),
            "perp2_frac": np.abs(g[["V_perp2", "M_perp1"]].to_numpy()).max() / vsc,
            "dM_minus_V": e_minus, "dM_plus_V": e_plus,
            "n_last": int(g["n_free_body"].iloc[-1]),
        })
    chk = pd.DataFrame(rows)
    return chk

## Fuselage

In [ ]:
CONFIG_PATH = Path("..","config","fuselage.json")
CONFIG = json.loads(CONFIG_PATH.read_text())["bodies"]

res = section_loads(df, CONFIG)
chk = check(res)